# Tutorial: Netherlands aFRR and dual pricing

Audience:
- operators, optimizers, and quant teams evaluating the Netherlands full-stack surface

Prerequisites:
- a repo checkout with the `dl` environment available
- familiarity with day-ahead scheduling and quarter-hour imbalance settlement

Learning goals:
- run the Dutch single-asset `da_plus_afrr` path
- inspect how the Dutch dual-price imbalance feed is normalized
- run the Dutch portfolio `schedule_revision` + `da_plus_afrr` full-stack path


In [ ]:
from __future__ import annotations

from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd

from euroflex_bess_lab.analytics.reporting import load_report_summary
from euroflex_bess_lab.backtesting.engine import run_walk_forward
from euroflex_bess_lab.config import load_config

REPO_ROOT = Path.cwd()
REPO_ROOT

## Step 1 - Run the Netherlands single-asset aFRR path

This uses the frozen Dutch aFRR benchmark curves that mirror the public repo examples. The output is a benchmark-grade reserve-aware dispatch, not a live submission payload.


In [ ]:
with TemporaryDirectory() as tmp:
    config = load_config(REPO_ROOT / "examples/configs/reserve/netherlands_da_plus_afrr_base.yaml")
    config.artifacts.root_dir = Path(tmp)
    result = run_walk_forward(config)
    summary = load_report_summary(result.output_dir)
    single_asset_summary = {
        key: summary[key]
        for key in [
            "market_id",
            "workflow",
            "run_scope",
            "reserve_product_id",
            "reserve_capacity_revenue_eur",
            "reserve_activation_revenue_eur",
            "total_pnl_eur",
        ]
    }
    dispatch_preview = result.site_dispatch[
        [
            "timestamp_local",
            "net_export_mw",
            "soc_mwh",
            "afrr_up_reserved_mw",
            "afrr_down_reserved_mw",
        ]
    ].head(8)
single_asset_summary, dispatch_preview

## Step 2 - Inspect Dutch dual-price imbalance behavior

The Dutch imbalance feed carries shortage and surplus prices separately. That is the key market-specific signal that makes TenneT normalization different from the Belgian single-price path.


In [ ]:
imbalance = pd.read_csv(REPO_ROOT / "examples/data/netherlands_imbalance_prices.csv")
imbalance["dual_price_spread_eur_per_mwh"] = (
    imbalance["imbalance_shortage_price_eur_per_mwh"] - imbalance["imbalance_surplus_price_eur_per_mwh"]
)
imbalance[
    [
        "timestamp_local",
        "price_eur_per_mwh",
        "imbalance_shortage_price_eur_per_mwh",
        "imbalance_surplus_price_eur_per_mwh",
        "regulation_state",
        "regulating_condition",
        "dual_price_spread_eur_per_mwh",
    ]
].head(8)

## Step 3 - Run the Netherlands promoted full-stack path

This is the Dutch analogue of the Belgium full-stack story: portfolio, shared POI, `schedule_revision`, and `base_workflow: da_plus_afrr`.


In [ ]:
with TemporaryDirectory() as tmp:
    config = load_config(REPO_ROOT / "examples/configs/canonical/netherlands_full_stack.yaml")
    config.artifacts.root_dir = Path(tmp)
    result = run_walk_forward(config)
    summary = load_report_summary(result.output_dir)
    full_stack_summary = {
        key: summary[key]
        for key in [
            "market_id",
            "workflow",
            "base_workflow",
            "run_scope",
            "asset_count",
            "total_pnl_eur",
            "reserve_capacity_revenue_eur",
            "reserve_activation_revenue_eur",
        ]
    }
    revision_preview = pd.read_parquet(result.output_dir / "revision_schedule.parquet")[
        [
            "timestamp_local",
            "schedule_version",
            "lock_state",
            "net_export_mw",
            "afrr_up_reserved_mw",
            "afrr_down_reserved_mw",
        ]
    ].head(8)
full_stack_summary, revision_preview

## Takeaways

- Dutch full-stack support now shares the same public-core artifact and export contracts as the Belgium path.
- The TenneT imbalance connector matters even when the main workflow is reserve-aware, because Dutch operator workflows still care about provenance and dual-price context.
- Both Belgium and Netherlands aFRR remain expected-value benchmarks, so reserve activation should be interpreted with the repo's published limitations in mind.
